# Extract a dataset of activations

`tl.extract_dataset` runs a stimulus set through a model in batches and collects named-layer activations -- in memory by default, or as a self-describing on-disk artifact when `output_dir=` is given. This notebook uses a seeded tiny model so it runs anywhere on CPU.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
from torch import nn

import torchlens as tl


def tiny_model(seed: int = 0) -> nn.Module:
    """Return a deterministic three-layer MLP for the 5-minute track."""

    torch.manual_seed(seed)
    return nn.Sequential(
        nn.Linear(4, 8),
        nn.ReLU(),
        nn.Linear(8, 2),
    )


torch.manual_seed(0)
model = tiny_model(seed=7).eval()
stimuli = torch.randn(10, 4)

In-memory mode returns a dict of stacked activations, one tensor per requested layer, ordered like the stimuli.

In [ ]:
features = tl.extract_dataset(model, stimuli, layers=["relu_1_2"], batch_size=4, progress=False)
print({name: tuple(tensor.shape) for name, tensor in features.items()})

Disk mode writes atomic shards plus a `manifest.json` (run signature, per-site identity, stimulus ordering, dtypes, TorchLens version) and returns the shard paths. `load_extraction` reads the artifact back with its metadata.

In [ ]:
from torchlens.dataset_extraction import load_extraction

with TemporaryDirectory() as tmpdir:
    out_dir = Path(tmpdir) / "tiny_extraction"
    shard_paths = tl.extract_dataset(
        model,
        stimuli,
        layers=["relu_1_2"],
        batch_size=4,
        output_dir=out_dir,
        progress=False,
    )
    print("shards written:", len(shard_paths))
    loaded = load_extraction(out_dir)
    activations = loaded.activations["relu_1_2"]
    print("loaded:", tuple(activations.shape))
    assert torch.equal(activations, features["relu_1_2"])

Interrupted runs resume from the last completed shard with `resume=True` (a run-signature check refuses mismatched resumes), and `stimulus_ids=` records your own stimulus naming in the manifest.